# 🩺 MedGemma Starter Notebook

This notebook provides a basic template for loading and running inference with **MedGemma** using the Hugging Face `transformers` library.

### 1. Setup & Authentication
Ensure you have installed the required libraries:
```bash
pip install transformers accelerate bitsandbytes
```

In [ ]:
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Replace with your token if not already logged in via CLI
# login("hf_...")

### 2. Load Model & Tokenizer
We use 4-bit quantization to fit the model on consumer GPUs.

In [ ]:
model_id = "google/med-gemma-7b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

### 3. Basic Inference
Let's ask a simple medical reasoning question.

In [ ]:
prompt = "A 65-year-old male presents with a sudden onset of sharp chest pain and shortness of breath. He has a history of smoking and hypertension. What are the most likely differential diagnoses?"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=250)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### 4. RAG Integration (Concept)
In a real hackathon project, you would retrieve patient data from the Neo4j Knowledge Graph and inject it into the prompt here.